In [6]:
import math
import os
import re

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold
from tqdm.auto import tqdm

PATH = os.path.dirname(os.path.abspath(__name__))
GRAPH_PATH = os.path.join(PATH, "graphs/graph_31444_nodes.gexf")
DATASET_PATH = os.path.join(PATH, "datasets/data.csv")


/home/kit-matthewson/Documents/exeter/masters/coursework/systems/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def as_label(value):
    """Convert labels read from CSV/GEXF into 0 (benign) or 1 (malicious)."""
    if isinstance(value, bool):
        return int(value)
    return int(str(value).strip().lower() in {"bad", "true", "1", "malicious"})


def metrics(y_true, y_pred):
    return {
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "accuracy": accuracy_score(y_true, y_pred),
    }


def summarise(results):
    return pd.DataFrame(results).mean().to_frame().T.round(3)


def url_features(url):
    url = str(url)
    
    letters = sum(char.isalpha() for char in url)
    digits = sum(char.isdigit() for char in url)
    special = sum(not char.isalnum() for char in url)
    counts = {char: url.count(char) for char in ".-/@:?!=_&%#;"}
    tokens = re.findall(r"[a-z0-9]+", url.lower())
    
    probabilities = [url.count(char) / len(url) for char in set(url)]
    entropy = -sum(probability * math.log2(probability) for probability in probabilities)
    
    return [
        len(url), len(tokens), len(url.split(".")), letters, digits, special,
        digits / max(len(url), 1), special / max(len(url), 1), entropy,
        int("@" in url), int("//" in url[8:]), int("https" in url.lower()),
        int(any(token in url.lower() for token in ("login", "verify", "secure", "account"))),
        counts["."], counts["-"], counts["/"], counts["?"], counts["="], counts["&"],
    ]


def evaluate_random_forest(urls, labels, splitter):
    features = np.asarray([
        url_features(url)
        for url in tqdm(urls, desc="RF: extracting URL features", unit="url")
    ])
    
    results = []
    folds = splitter.split(features, labels)
    for fold, (train, test) in enumerate(
        tqdm(folds, total=splitter.get_n_splits(), desc="RF: cross-validation", unit="fold"),
        start=1,
    ):
        model = RandomForestClassifier(
            n_estimators=300, random_state=42, n_jobs=-1, class_weight="balanced"
        )
        model.fit(features[train], labels[train])
        results.append(metrics(labels[test], model.predict(features[test])))
    return results


In [ ]:
def evaluate_pol(graph, url_nodes, labels, splitter, epochs=5):
    compatibility = np.asarray([[0.9, 0.1], [0.1, 0.9]])
    all_nodes = list(graph.nodes)
    node_index = {node: index for index, node in enumerate(all_nodes)}
    results = []
    folds = splitter.split(url_nodes, labels)
    for fold, (train, test) in enumerate(
        tqdm(folds, total=splitter.get_n_splits(), desc="POL: cross-validation", unit="fold"),
        start=1,
    ):
        beliefs = np.full((len(all_nodes), 2), 0.5)
        observed = {url_nodes[index] for index in train}
        for index in train:
            beliefs[node_index[url_nodes[index]]] = [1.0, 0.0] if labels[index] == 0 else [0.0, 1.0]

        for epoch in tqdm(range(epochs), desc=f"POL fold {fold}: epochs", unit="epoch", leave=False):
            updated = beliefs.copy()
            for node in tqdm(all_nodes, desc=f"POL fold {fold}: nodes", unit="node", leave=False):
                if node in observed:
                    continue
                neighbours = list(graph.neighbors(node))
                if neighbours:
                    messages = [compatibility @ beliefs[node_index[nbr]] for nbr in neighbours]
                    updated[node_index[node]] = np.mean(messages, axis=0)
                    updated[node_index[node]] /= updated[node_index[node]].sum()
            beliefs = updated

        predictions = [int(beliefs[node_index[url], 1] >= beliefs[node_index[url], 0]) for url in url_nodes]
        results.append(metrics(labels[test], np.asarray(predictions)[test]))
    return results


In [9]:
# Use the same labelled URL population and folds for both baselines.
graph = nx.read_gexf(GRAPH_PATH)
url_nodes = [node for node, data in graph.nodes(data=True) if data.get("type") == "url" and data.get("malicious") is not None]
url_nodes = np.asarray(url_nodes)
labels = np.asarray([as_label(graph.nodes[node]["malicious"]) for node in url_nodes])
urls = url_nodes.copy()
splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rf_results = evaluate_random_forest(urls, labels, splitter)
pol_results = evaluate_pol(graph, url_nodes, labels, splitter)

results = pd.DataFrame({
    "random_forest": summarise(rf_results).iloc[0],
    "POL": summarise(pol_results).iloc[0],
}).T[["precision", "recall", "f1", "accuracy"]]
results


POL: cross-validation:   0%|          | 0/5 [00:00<?, ?fold/s]







































POL: cross-validation:  20%|██        | 1/5 [00:03<00:13,  3.50s/fold]







































POL: cross-validation:  40%|████      | 2/5 [00:07<00:10,  3.51s/fold]







































POL: cross-validation:  60%|██████    | 3/5 [00:10<00:07,  3.51s/fold]








































POL: cross-validation:  80%|████████  | 4/5 [00:14<00:03,  3.52s/fold]







































POL: cross-validation: 100%|██████████| 5/5 [00:17<00:00,  3.52s/fold]


,precision,recall,f1,accuracy
random_forest,0.815,0.821,0.818,0.817
POL,0.928,0.885,0.906,0.908
